# AnyProjector v0.9.7 — Phase 2 Anti-Plateau Alignment

**Upgrades from v0.9.6:**
- Dropout 0.1 in Q-Former (anti neuron memorization)
- SpecAugment on mel spectrogram (anti audio memorization)
- MoCo-style negative queue 4096 (anti negative exhaustion)
- Fixed early stopping: val-loss-only, no overfit ratio

**Pipeline:**
```
Audio → SpecAugment → Whisper(frozen) → Projector+Dropout(train) → mean_pool → audio_vec
Text  → embed_layer(frozen)                                      → mean_pool → text_vec

Loss = cosine(audio_vec, text_vec)
     + InfoNCE(audio_vec, [text_batch + text_queue_4096])
```


In [ ]:
!pip install -q transformers datasets torch accelerate hf_transfer huggingface_hub matplotlib
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('Done')

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('No GPU!')
print(f'PyTorch: {torch.__version__}')

In [ ]:
from huggingface_hub import login
login()

## Config

In [ ]:
ENCODER_ID    = "openai/whisper-medium"
LLM_ID        = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID    = "doof-ferb/vlsp2020_vinai_100h"
NUM_SHARDS    = 35

NUM_EPOCHS  = 50
BATCH_SIZE  = 400
LR          = 5e-4
GRAD_ACCUM  = 2
SAVE_DIR    = "checkpoints/phase2/v097_antiplateau"
PATIENCE    = 10
MIN_DELTA   = 0.005
RESUME_FROM = None
NUM_WORKERS = 4
PRELOAD_RAM = True

# Q-Former
QFORMER_LAYERS = 4
QFORMER_HEADS  = 16
DROPOUT        = 0.1

# Loss
CONTRASTIVE_TEMP   = 0.07
COSINE_WEIGHT      = 1.0
CONTRASTIVE_WEIGHT = 1.0

# MoCo Queue
QUEUE_SIZE = 4096

# SpecAugment
SPEC_FREQ_MASKS = 2
SPEC_TIME_MASKS = 2
SPEC_FREQ_WIDTH = 27
SPEC_TIME_WIDTH = 100

UNFREEZE_ENCODER_LAYERS = 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_SAVE = f'/content/drive/MyDrive/AnyProjector/{SAVE_DIR}'
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f'Drive: {DRIVE_SAVE}')

## Load Dataset

In [ ]:
import time
from datasets import load_dataset as hf_load_dataset

print(f'Loading {NUM_SHARDS}/35 shards from {DATASET_ID}...')
t0 = time.time()
shard_files = [f'data/train-{i:05d}-of-00035.parquet' for i in range(NUM_SHARDS)]
ds = hf_load_dataset(DATASET_ID, data_files=shard_files, split='train', token=True, verification_mode='no_checks')
print(f'Loaded {len(ds)} samples in {(time.time()-t0)/60:.1f} min')
print(f'Columns: {ds.column_names}')
print(f'Example: {ds[0]["transcription"][:80]}...')

## AnyProjector Q-Former (with Dropout)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class QFormerLayer(nn.Module):
    """Q-Former layer with Dropout."""

    def __init__(self, qformer_dim, encoder_dim, num_heads=8, ffn_ratio=4, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, batch_first=True, dropout=dropout)
        self.self_attn_norm = nn.LayerNorm(qformer_dim)
        self.self_attn_drop = nn.Dropout(dropout)

        self.cross_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, kdim=encoder_dim, vdim=encoder_dim, batch_first=True, dropout=dropout)
        self.cross_attn_norm = nn.LayerNorm(qformer_dim)
        self.cross_attn_drop = nn.Dropout(dropout)

        ffn_hidden = qformer_dim * ffn_ratio
        self.ffn = nn.Sequential(
            nn.Linear(qformer_dim, ffn_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_hidden, qformer_dim), nn.Dropout(dropout),
        )
        self.ffn_norm = nn.LayerNorm(qformer_dim)

    def forward(self, queries, encoder_out, encoder_mask=None):
        q = self.self_attn_norm(queries)
        q, _ = self.self_attn(q, q, q)
        queries = queries + self.self_attn_drop(q)

        q = self.cross_attn_norm(queries)
        q, _ = self.cross_attn(query=q, key=encoder_out, value=encoder_out, key_padding_mask=encoder_mask)
        queries = queries + self.cross_attn_drop(q)

        queries = queries + self.ffn(self.ffn_norm(queries))
        return queries


class AnyProjector(nn.Module):
    """Q-Former Projector with Dropout."""

    def __init__(self, encoder_dim, llm_dim, num_queries=64, qformer_dim=768,
                 num_layers=2, num_heads=8, dropout=0.1):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.llm_dim = llm_dim
        self.num_queries = num_queries
        self.qformer_dim = qformer_dim
        self.pre_proj = nn.Sequential(
            nn.Linear(encoder_dim, encoder_dim), nn.GELU(), nn.LayerNorm(encoder_dim),
        )
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, qformer_dim) * 0.02)
        self.layers = nn.ModuleList([
            QFormerLayer(qformer_dim, encoder_dim, num_heads, dropout=dropout) for _ in range(num_layers)
        ])
        self.output_norm = nn.LayerNorm(qformer_dim)
        self.output_proj = nn.Sequential(
            nn.Linear(qformer_dim, llm_dim), nn.Dropout(dropout),
        )

    def forward(self, encoder_output, encoder_mask=None):
        B = encoder_output.shape[0]
        encoder_output = self.pre_proj(encoder_output)
        queries = self.query_tokens.expand(B, -1, -1)
        for layer in self.layers:
            queries = layer(queries, encoder_output, encoder_mask)
        queries = self.output_norm(queries)
        return self.output_proj(queries)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self):
        return f"AnyProjector(queries={self.num_queries}, layers={len(self.layers)}, params={self.count_parameters():,})"

## Imports + Config + Dataset

In [ ]:
import gc, logging, math, time, random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset as hf_load_dataset
from torch.utils.data import Dataset, DataLoader

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s",
                    datefmt="%H:%M:%S", force=True)
logger = logging.getLogger("phase2")


@dataclass
class Phase2Config:
    encoder_id: str = ENCODER_ID
    llm_id: str = LLM_ID
    datasets: tuple = ((DATASET_ID, 'transcription', 'auto_split'),)
    max_samples_per_dataset: int = 0
    auto_val_ratio: float = 0.1
    sample_rate: int = 16000
    max_audio_seconds: float = 30.0
    num_queries: int = 64
    qformer_dim: int = 768
    qformer_layers: int = QFORMER_LAYERS
    qformer_heads: int = QFORMER_HEADS
    dropout: float = DROPOUT
    unfreeze_encoder_layers: int = UNFREEZE_ENCODER_LAYERS
    num_epochs: int = NUM_EPOCHS
    batch_size: int = BATCH_SIZE
    learning_rate: float = LR
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    gradient_accumulation_steps: int = GRAD_ACCUM
    save_dir: str = SAVE_DIR
    save_every: int = 5
    early_stopping_patience: int = PATIENCE
    early_stopping_min_delta: float = MIN_DELTA
    resume_from: str = RESUME_FROM
    num_workers: int = NUM_WORKERS
    preload_ram: bool = PRELOAD_RAM
    max_text_tokens: int = 128
    contrastive_temp: float = CONTRASTIVE_TEMP
    cosine_weight: float = COSINE_WEIGHT
    contrastive_weight: float = CONTRASTIVE_WEIGHT
    queue_size: int = QUEUE_SIZE
    spec_freq_masks: int = SPEC_FREQ_MASKS
    spec_time_masks: int = SPEC_TIME_MASKS
    spec_freq_width: int = SPEC_FREQ_WIDTH
    spec_time_width: int = SPEC_TIME_WIDTH


class Phase2Dataset(Dataset):
    def __init__(self, entries, sample_rate=16000, max_audio_seconds=30.0, preload_ram=False):
        self.entries = entries
        self.sample_rate = sample_rate
        self.max_samples = int(max_audio_seconds * sample_rate)
        logger.info(f"Dataset: {len(entries)} samples")
        self.cache = None
        if preload_ram:
            logger.info("Preloading audio to RAM...")
            self.cache = []
            for i, entry in enumerate(entries):
                self.cache.append(self._process_audio(entry["audio"]))
                if (i + 1) % 1000 == 0: logger.info(f"  preloaded {i+1}/{len(entries)}")
            ram_mb = sum(w.nbytes for w in self.cache) / 1024**2
            logger.info(f"  Done! {len(self.cache)} samples ({ram_mb:.0f} MB RAM)")

    def __len__(self): return len(self.entries)

    def _process_audio(self, audio_feature):
        waveform = torch.from_numpy(audio_feature["array"].astype(np.float32))
        sr = audio_feature["sampling_rate"]
        if waveform.dim() > 1: waveform = waveform.mean(dim=-1)
        if sr != self.sample_rate:
            import torchaudio
            waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform.unsqueeze(0)).squeeze(0)
        if waveform.shape[0] > self.max_samples: waveform = waveform[:self.max_samples]
        return waveform

    def __getitem__(self, idx):
        waveform = self.cache[idx] if self.cache else self._process_audio(self.entries[idx]["audio"])
        return {"waveform": waveform, "transcript": self.entries[idx]["transcript"]}


def collate_fn(batch):
    waveforms = [s["waveform"] for s in batch]
    transcripts = [s["transcript"] for s in batch]
    lengths = torch.tensor([w.shape[0] for w in waveforms])
    waveforms_padded = nn.utils.rnn.pad_sequence(waveforms, batch_first=True, padding_value=0.0)
    return {"waveforms": waveforms_padded, "lengths": lengths, "transcripts": transcripts}

## Trainer v0.9.7 — Dropout + SpecAugment + MoCo Queue

```
Loss = cosine(audio, text)  +  InfoNCE(audio, text_batch + text_queue)
       ^aligned               ^discriminative (4496 negatives!)
```

In [ ]:
class Phase2Trainer:
    def __init__(self, config: Phase2Config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        logger.info(f"Device: {self.device}")
        self.encoder = self.projector = self.embed_layer = None
        self.tokenizer = self.processor = self.optimizer = self.scheduler = None
        self.log_temp = None
        self.global_step = 0
        self.start_epoch = 0
        # MoCo Queue
        self.text_queue = None  # (queue_size, llm_dim)
        self.queue_ptr = 0

    # ========== SpecAugment ==========
    def spec_augment(self, mel_spec):
        """Apply SpecAugment on mel spectrogram (B, n_mels, T).
        Random zero-mask freq bands and time segments.
        Only during training."""
        cfg = self.config
        B, n_mels, T = mel_spec.shape
        mel_spec = mel_spec.clone()
        for i in range(B):
            # Frequency masking
            for _ in range(cfg.spec_freq_masks):
                f = random.randint(0, cfg.spec_freq_width)
                f0 = random.randint(0, max(1, n_mels - f))
                mel_spec[i, f0:f0+f, :] = 0
            # Time masking
            for _ in range(cfg.spec_time_masks):
                t = random.randint(0, cfg.spec_time_width)
                t0 = random.randint(0, max(1, T - t))
                mel_spec[i, :, t0:t0+t] = 0
        return mel_spec

    # ========== MoCo Queue ==========
    @torch.no_grad()
    def _enqueue(self, text_norm):
        """Push text embeddings into FIFO queue."""
        B = text_norm.shape[0]
        if self.text_queue is None:
            dim = text_norm.shape[1]
            self.text_queue = torch.zeros(self.config.queue_size, dim, device=self.device)
            logger.info(f"  Queue initialized: {self.config.queue_size} x {dim} ({self.text_queue.nbytes/1024**2:.1f}MB)")
        ptr = self.queue_ptr
        space = self.config.queue_size - ptr
        if B <= space:
            self.text_queue[ptr:ptr+B] = text_norm
        else:
            self.text_queue[ptr:] = text_norm[:space]
            self.text_queue[:B-space] = text_norm[space:]
        self.queue_ptr = (ptr + B) % self.config.queue_size

    def setup_models(self):
        from transformers import WhisperModel, WhisperProcessor, AutoModelForCausalLM, AutoTokenizer, AutoConfig

        logger.info(f"Loading encoder: {self.config.encoder_id}")
        self.processor = WhisperProcessor.from_pretrained(self.config.encoder_id)
        whisper_full = WhisperModel.from_pretrained(self.config.encoder_id)
        self.encoder = whisper_full.encoder.eval()
        del whisper_full; gc.collect()
        for p in self.encoder.parameters(): p.requires_grad = False
        if self.config.unfreeze_encoder_layers > 0:
            for layer in self.encoder.layers[-self.config.unfreeze_encoder_layers:]:
                for p in layer.parameters(): p.requires_grad = True
                layer.train()
        encoder_dim = self.encoder.config.d_model
        logger.info(f"  encoder_dim={encoder_dim}")

        logger.info(f"Extracting embed_layer from: {self.config.llm_id}")
        llm_config = AutoConfig.from_pretrained(self.config.llm_id)
        llm_dim = llm_config.hidden_size
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.llm_id)
        if self.tokenizer.pad_token is None: self.tokenizer.pad_token = self.tokenizer.eos_token
        llm_temp = AutoModelForCausalLM.from_pretrained(self.config.llm_id, torch_dtype=torch.float32)
        self.embed_layer = llm_temp.get_input_embeddings().to(self.device).eval()
        for p in self.embed_layer.parameters(): p.requires_grad = False
        del llm_temp; gc.collect(); torch.cuda.empty_cache()
        logger.info(f"  llm_dim={llm_dim}")

        self.projector = AnyProjector(
            encoder_dim=encoder_dim, llm_dim=llm_dim,
            num_queries=self.config.num_queries, qformer_dim=self.config.qformer_dim,
            num_layers=self.config.qformer_layers, num_heads=self.config.qformer_heads,
            dropout=self.config.dropout,
        ).to(self.device).train()
        logger.info(f"Projector: {self.projector.count_parameters():,} params (dropout={self.config.dropout})")

        self.log_temp = nn.Parameter(torch.tensor(math.log(1.0 / self.config.contrastive_temp), device=self.device))
        self.encoder.to(self.device)

        if torch.cuda.is_available():
            logger.info(f"  VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

    def setup_optimizer(self, total_steps):
        self._trainable_params = list(self.projector.parameters()) + [self.log_temp]
        if self.config.unfreeze_encoder_layers > 0:
            self._trainable_params += [p for p in self.encoder.parameters() if p.requires_grad]
        total_trainable = sum(p.numel() for p in self._trainable_params)
        logger.info(f"Trainable: {total_trainable:,} params")

        self.optimizer = torch.optim.AdamW(self._trainable_params, lr=self.config.learning_rate, weight_decay=self.config.weight_decay)
        warmup_steps = int(total_steps * self.config.warmup_ratio)
        def lr_lambda(step):
            if step < warmup_steps: return float(step) / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        logger.info(f"Optimizer: AdamW lr={self.config.learning_rate}, warmup={warmup_steps}/{total_steps}")

    def process_batch(self, batch, is_train=True):
        """Process batch with SpecAugment + MoCo Queue."""
        waveforms = batch["waveforms"]
        transcripts = batch["transcripts"]

        # --- Encoder mask ---
        encoder_seq_len = 1500
        samples_per_token = (self.config.max_audio_seconds * self.config.sample_rate) / encoder_seq_len
        encoder_mask = torch.zeros(len(waveforms), encoder_seq_len, dtype=torch.bool, device=self.device)
        for i, w in enumerate(waveforms):
            real_tokens = min(encoder_seq_len, int(w.shape[0] / samples_per_token))
            encoder_mask[i, real_tokens:] = True

        # --- 1. Audio → Mel → SpecAugment → Encoder → Projector → mean pool ---
        with torch.set_grad_enabled(self.config.unfreeze_encoder_layers > 0):
            audio_inputs = self.processor(
                [w.numpy() for w in waveforms], sampling_rate=self.config.sample_rate,
                return_tensors="pt", padding="max_length",
            )
            input_features = audio_inputs.input_features.to(self.device)  # (B, 128, 3000)

            # SpecAugment (training only)
            if is_train:
                input_features = self.spec_augment(input_features)

            encoder_output = self.encoder(input_features).last_hidden_state

        proj_out = self.projector(encoder_output, encoder_mask)  # (B, 64, llm_dim)
        audio_vec = proj_out.mean(dim=1)  # (B, llm_dim)

        # --- 2. Text → embed_layer → mean pool ---
        with torch.no_grad():
            text_tokens = self.tokenizer(
                transcripts, return_tensors="pt", padding=True,
                truncation=True, max_length=self.config.max_text_tokens,
                add_special_tokens=False,
            ).to(self.device)
            text_embeds = self.embed_layer(text_tokens.input_ids)
            mask = text_tokens.attention_mask.unsqueeze(-1).float()
            text_vec = (text_embeds * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        # --- 3. Normalize ---
        audio_norm = F.normalize(audio_vec, dim=-1)
        text_norm = F.normalize(text_vec, dim=-1)

        # --- 4. Cosine alignment loss ---
        cos_sim = (audio_norm * text_norm).sum(dim=-1)
        cosine_loss = (1 - cos_sim).mean()

        # --- 5. Contrastive loss with MoCo Queue ---
        temp = torch.exp(self.log_temp).clamp(max=100.0)
        B = audio_norm.shape[0]
        labels = torch.arange(B, device=self.device)

        # audio → text (with queue negatives)
        logits_a2t = audio_norm @ text_norm.T * temp  # (B, B)
        if is_train and self.text_queue is not None and self.queue_ptr > 0:
            # Use filled portion of queue
            queue_size = min(self.config.queue_size, self.queue_ptr + B * 10)  # progressive fill
            q = self.text_queue[:queue_size].clone().detach()  # (Q, dim) — clone to avoid inplace grad error
            logits_a2q = audio_norm @ q.T * temp  # (B, Q)
            logits_a2t_full = torch.cat([logits_a2t, logits_a2q], dim=1)  # (B, B+Q)
            loss_a2t = F.cross_entropy(logits_a2t_full, labels)
        else:
            loss_a2t = F.cross_entropy(logits_a2t, labels)

        # text → audio (in-batch only, no queue for audio side)
        logits_t2a = text_norm @ audio_norm.T * temp  # (B, B)
        loss_t2a = F.cross_entropy(logits_t2a, labels)

        contrastive_loss = (loss_a2t + loss_t2a) / 2

        # --- Enqueue text embeddings (training only) ---
        if is_train:
            self._enqueue(text_norm.detach())

        # --- Combined loss ---
        total_loss = (self.config.cosine_weight * cosine_loss +
                      self.config.contrastive_weight * contrastive_loss)

        self._last_metrics = {
            "cosine_loss": cosine_loss.item(),
            "contrastive_loss": contrastive_loss.item(),
            "mean_cos_sim": cos_sim.mean().item(),
            "temperature": (1.0 / temp).item(),
        }
        return total_loss

    def log_gradient_diagnostics(self):
        proj_grad_norm = 0.0; proj_count = 0; proj_zero = 0
        for name, param in self.projector.named_parameters():
            if param.requires_grad and param.grad is not None:
                proj_grad_norm += param.grad.data.norm(2).item() ** 2; proj_count += 1
                if param.grad.data.norm(2).item() < 1e-8: proj_zero += 1
            elif param.requires_grad: proj_zero += 1; proj_count += 1
        qgn = self.projector.query_tokens.grad.data.norm(2).item() if self.projector.query_tokens.grad is not None else 0.0
        return {"proj_grad": proj_grad_norm**0.5, "query_grad": qgn, "zeros": f"{proj_zero}/{proj_count}"}

    def log_embedding_diagnostics(self, batch):
        self.projector.eval()
        with torch.no_grad():
            waveforms = batch["waveforms"]; transcripts = batch["transcripts"]
            if waveforms.shape[0] < 2: return {}
            proj_vecs, text_vecs = [], []
            for i in range(min(2, waveforms.shape[0])):
                ai = self.processor(waveforms[i].numpy(), sampling_rate=self.config.sample_rate, return_tensors="pt", padding="max_length")
                eo = self.encoder(ai.input_features.to(self.device)).last_hidden_state
                proj_vecs.append(self.projector(eo).mean(dim=1).flatten())
                tt = self.tokenizer(transcripts[i], return_tensors="pt", add_special_tokens=False).to(self.device)
                text_vecs.append(self.embed_layer(tt.input_ids).mean(dim=1).flatten())
            c_at = F.cosine_similarity(proj_vecs[0].unsqueeze(0), text_vecs[0].unsqueeze(0)).item()
            c_pp = F.cosine_similarity(proj_vecs[0].unsqueeze(0), proj_vecs[1].unsqueeze(0)).item()
        self.projector.train()
        return {"cos_at": c_at, "cos_pp": c_pp, "norm": sum(v.norm().item() for v in proj_vecs)/2}

    def _vram(self):
        if torch.cuda.is_available():
            a = torch.cuda.memory_allocated()/1024**3
            t = torch.cuda.get_device_properties(0).total_memory/1024**3
            return f"{a:.1f}/{t:.1f}GB"
        return "N/A"

    def train(self, preloaded_dataset=None):
        config = self.config
        self.setup_models()

        # --- Dataset ---
        train_entries, val_entries = [], []
        if preloaded_dataset is not None:
            logger.info(f"Using preloaded dataset: {len(preloaded_dataset)} samples")
            transcript_field = config.datasets[0][1]
            limit = config.max_samples_per_dataset
            total = len(preloaded_dataset)
            if limit > 0 and total > limit: preloaded_dataset = preloaded_dataset.select(range(limit)); total = limit
            t_extract = time.time()
            audios = preloaded_dataset["audio"]; transcripts = preloaded_dataset[transcript_field]
            entries = [{"audio": audios[i], "transcript": transcripts[i]} for i in range(total)]
            del audios, transcripts
            logger.info(f"  Extracted {len(entries)} in {time.time()-t_extract:.1f}s")
            rng = random.Random(42)
            val_count = max(1, int(len(entries) * config.auto_val_ratio))
            indices = list(range(len(entries))); rng.shuffle(indices)
            for idx in indices[:len(entries)-val_count]: train_entries.append(entries[idx])
            for idx in indices[len(entries)-val_count:]: val_entries.append(entries[idx])
            logger.info(f"  -> {len(train_entries)} TRAIN + {len(val_entries)} VAL")
            del preloaded_dataset
        else:
            from datasets import concatenate_datasets
            for ds_config in config.datasets:
                ds_name, tf, mode = ds_config
                available = {}
                for sn in ["train","validation","test"]:
                    try: available[sn] = hf_load_dataset(ds_name, split=sn, token=True)
                    except: pass
                if not available: continue
                merged = concatenate_datasets(list(available.values()))
                entries = [{"audio": merged[i]["audio"], "transcript": merged[i][tf]} for i in range(len(merged))]
                rng = random.Random(42)
                vc = max(1, int(len(entries) * config.auto_val_ratio))
                idx = list(range(len(entries))); rng.shuffle(idx)
                for j in idx[:len(entries)-vc]: train_entries.append(entries[j])
                for j in idx[len(entries)-vc:]: val_entries.append(entries[j])

        logger.info(f"Dataset: {len(train_entries)} train + {len(val_entries)} val")
        train_dataset = Phase2Dataset(train_entries, config.sample_rate, config.max_audio_seconds, config.preload_ram)
        val_dataset = Phase2Dataset(val_entries, config.sample_rate, config.max_audio_seconds, config.preload_ram)
        nw = config.num_workers
        train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=nw, pin_memory=True, persistent_workers=nw>0)
        val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=nw, pin_memory=True, persistent_workers=nw>0)
        logger.info(f"Batches/epoch: {len(train_loader)}")

        steps_per_epoch = math.ceil(len(train_loader) / config.gradient_accumulation_steps)
        total_steps = steps_per_epoch * config.num_epochs
        self.setup_optimizer(total_steps)
        if config.resume_from: self.load_checkpoint(config.resume_from)

        # Banner
        logger.info("")
        logger.info("=" * 60)
        logger.info("  PHASE 2 v0.9.7 — ANTI-PLATEAU ALIGNMENT")
        logger.info(f"  Projector:  {self.projector.count_parameters():,} params (dropout={config.dropout})")
        logger.info(f"  Loss:       Cosine + InfoNCE")
        logger.info(f"  SpecAugment: freq={config.spec_freq_masks}x{config.spec_freq_width}, time={config.spec_time_masks}x{config.spec_time_width}")
        logger.info(f"  MoCo Queue: {config.queue_size} text embeddings")
        logger.info(f"  Batch:      {config.batch_size}x{config.gradient_accumulation_steps} = {config.batch_size*config.gradient_accumulation_steps} effective")
        logger.info(f"  LR:         {config.learning_rate}, Epochs: {config.num_epochs}")
        logger.info(f"  VRAM:       {self._vram()}")
        logger.info("=" * 60)
        logger.info("")

        best_val_loss = float("inf"); best_epoch = 0; patience_counter = 0; history = []

        for epoch in range(self.start_epoch, config.num_epochs):
            epoch_num = epoch + 1
            logger.info(f"--- Epoch {epoch_num}/{config.num_epochs} ---")

            # ===== TRAIN =====
            self.projector.train()
            t_sum = t_cos = t_con = 0.0; t_steps = 0
            self.optimizer.zero_grad()
            ep_start = time.time(); ep_grad = None

            for bi, batch in enumerate(train_loader):
                loss = self.process_batch(batch, is_train=True)
                lv = loss.float()
                (lv / config.gradient_accumulation_steps).backward()
                t_sum += lv.item(); t_cos += self._last_metrics["cosine_loss"]; t_con += self._last_metrics["contrastive_loss"]; t_steps += 1

                tot = len(train_loader); done = bi+1; filled = int(25*done/tot)
                bar = "#"*filled + "."*(25-filled)
                avg = t_sum/t_steps; cos_a = t_cos/t_steps; lr = self.optimizer.param_groups[0]["lr"]
                el = time.time()-ep_start
                print(f"\r  {bar} {done}/{tot} | loss={avg:.4f} cos={cos_a:.4f} | lr={lr:.2e} | {el:.0f}s | VRAM {self._vram()}", end="", flush=True)

                if (bi+1) % config.gradient_accumulation_steps == 0:
                    if ep_grad is None: ep_grad = self.log_gradient_diagnostics()
                    torch.nn.utils.clip_grad_norm_(self._trainable_params, config.max_grad_norm)
                    self.optimizer.step(); self.scheduler.step(); self.optimizer.zero_grad()
                    self.global_step += 1

            if t_steps % config.gradient_accumulation_steps != 0:
                torch.nn.utils.clip_grad_norm_(self._trainable_params, config.max_grad_norm)
                self.optimizer.step(); self.scheduler.step(); self.optimizer.zero_grad(); self.global_step += 1

            train_avg = t_sum/max(t_steps,1); train_cos = t_cos/max(t_steps,1); train_con = t_con/max(t_steps,1)
            elapsed = time.time()-ep_start; print()

            # ===== VAL =====
            logger.info(f"  Validating...")
            self.projector.eval()
            v_sum = v_cos = v_con = 0.0; v_steps = 0
            with torch.no_grad():
                for batch in val_loader:
                    loss = self.process_batch(batch, is_train=False)
                    v_sum += loss.float().item(); v_cos += self._last_metrics["cosine_loss"]
                    v_con += self._last_metrics["contrastive_loss"]; v_steps += 1
            val_avg = v_sum/max(v_steps,1); val_cos = v_cos/max(v_steps,1); val_con = v_con/max(v_steps,1)

            # ===== EPOCH SUMMARY =====
            ratio = val_avg / max(train_avg, 1e-8)
            history.append({"epoch": epoch_num, "train": train_avg, "val": val_avg,
                "train_cos": train_cos, "val_cos": val_cos, "train_con": train_con, "val_con": val_con,
                "lr": self.optimizer.param_groups[0]["lr"], "overfit_ratio": ratio,
                "grad_norm": ep_grad.get("proj_grad",0) if ep_grad else 0, "elapsed_s": elapsed})

            # Early stopping: val loss only (no overfit ratio)
            val_improved = val_avg < best_val_loss - config.early_stopping_min_delta
            if val_improved:
                best_val_loss = val_avg; best_epoch = epoch_num; patience_counter = 0
                tag = "BEST"; self.save_checkpoint("best", val_avg)
            else:
                patience_counter += 1
                tag = f"wait {patience_counter}/{config.early_stopping_patience}"

            mean_cos = self._last_metrics.get("mean_cos_sim", 0)
            temp_val = self._last_metrics.get("temperature", 0)
            q_fill = min(self.queue_ptr if self.text_queue is not None else 0, config.queue_size)

            logger.info(f"")
            logger.info(f"  Epoch {epoch_num:>2} | Train={train_avg:.4f} (cos={train_cos:.4f} con={train_con:.4f})")
            logger.info(f"          | Val  ={val_avg:.4f} (cos={val_cos:.4f} con={val_con:.4f}) {tag}")
            logger.info(f"          | cos(a,t)={mean_cos:.4f} temp={temp_val:.4f} ratio={ratio:.2f}x")
            logger.info(f"          | Queue: {q_fill}/{config.queue_size} | LR={self.optimizer.param_groups[0]['lr']:.2e} | {elapsed:.0f}s")

            if ep_grad:
                logger.info(f"  Grad: proj={ep_grad['proj_grad']:.4f} query={ep_grad['query_grad']:.4f} zeros={ep_grad['zeros']}")

            if epoch_num % 5 == 1 or epoch_num <= 3:
                try:
                    sb = next(iter(val_loader)); ed = self.log_embedding_diagnostics(sb)
                    if ed:
                        logger.info(f"  Embed: cos(a,t)={ed['cos_at']:.4f} cos(p1,p2)={ed['cos_pp']:.4f} norm={ed['norm']:.1f}")
                        if ed['cos_pp'] > 0.99: logger.info(f"  [!] COLLAPSE")
                        elif ed['cos_pp'] < 0.8: logger.info(f"  [OK] Differentiated")
                except Exception as e: logger.info(f"  (embed diag error: {e})")
            logger.info("")

            if epoch_num % config.save_every == 0: self.save_checkpoint(epoch_num, val_avg)
            if patience_counter >= config.early_stopping_patience:
                logger.info(f"Early stopping at epoch {epoch_num}. Best: E{best_epoch} val={best_val_loss:.4f}")
                self.save_checkpoint(f"early_e{epoch_num}", val_avg); break

        self.save_checkpoint("final", val_avg)

        # Final summary
        logger.info("")
        logger.info("=" * 60)
        logger.info("  TRAINING COMPLETE")
        logger.info(f"  Best: E{best_epoch} val={best_val_loss:.4f}")
        logger.info(f"  Last 10 epochs:")
        for h in history[-10:]:
            logger.info(f"  E{h['epoch']:>2} T={h['train']:.4f} V={h['val']:.4f} cos={h['val_cos']:.4f} con={h['val_con']:.4f}")
        logger.info("=" * 60)

        self._export_log(history)
        self._plot(history)
        self._backup_drive()

    def _backup_drive(self):
        import shutil
        src = Path(self.config.save_dir); dst = Path("/content/drive/MyDrive/AnyProjector") / self.config.save_dir
        if not src.exists() or not Path("/content/drive").exists(): return
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.is_file(): shutil.copy2(f, dst/f.name); logger.info(f"  -> Drive: {f.name}")
        try:
            from google.colab import runtime; runtime.unassign()
        except: pass

    def _export_log(self, history):
        import csv
        sd = Path(self.config.save_dir); sd.mkdir(parents=True, exist_ok=True)
        p = sd / "training_log.csv"
        fields = ["epoch","train","val","train_cos","val_cos","train_con","val_con","overfit_ratio","lr","grad_norm","elapsed_s"]
        with open(p, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
            for h in history: w.writerow({k: h.get(k,"") for k in fields})
        logger.info(f"Log: {p}")

    def _plot(self, history):
        try: import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        except: return
        sd = Path(self.config.save_dir); sd.mkdir(parents=True, exist_ok=True)
        ep = [h["epoch"] for h in history]
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle("Phase 2 v0.9.7 — Anti-Plateau", fontsize=14, fontweight="bold")
        ax = axes[0,0]; ax.plot(ep,[h["train"] for h in history],"b-o",ms=3,label="Train"); ax.plot(ep,[h["val"] for h in history],"r-s",ms=3,label="Val"); ax.set_title("Total Loss"); ax.legend(); ax.grid(True,alpha=0.3)
        ax = axes[0,1]; ax.plot(ep,[h["train_cos"] for h in history],"b-o",ms=3,label="Train"); ax.plot(ep,[h["val_cos"] for h in history],"r-s",ms=3,label="Val"); ax.set_title("Cosine Loss"); ax.legend(); ax.grid(True,alpha=0.3)
        ax = axes[1,0]; ax.plot(ep,[h["train_con"] for h in history],"b-o",ms=3,label="Train"); ax.plot(ep,[h["val_con"] for h in history],"r-s",ms=3,label="Val"); ax.set_title("Contrastive Loss"); ax.legend(); ax.grid(True,alpha=0.3)
        ax = axes[1,1]; ax.plot(ep,[h["overfit_ratio"] for h in history],"m-^",ms=3); ax.axhline(y=1.0,color="green",ls="--",alpha=0.5); ax.set_title("Overfit Ratio (informational)"); ax.grid(True,alpha=0.3)
        plt.tight_layout(); pp = sd/"training_curves.png"; fig.savefig(pp, dpi=150, bbox_inches="tight"); plt.close(fig)
        logger.info(f"Plot: {pp}")
        try: from IPython.display import display, Image; display(Image(filename=str(pp)))
        except: pass

    def save_checkpoint(self, tag, val_loss=None):
        sd = Path(self.config.save_dir); sd.mkdir(parents=True, exist_ok=True)
        ckpt = {
            "projector_state_dict": self.projector.state_dict(),
            "log_temp": self.log_temp.data.item(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict() if self.scheduler else None,
            "global_step": self.global_step,
            "epoch": tag if isinstance(tag, int) else -1,
            "val_loss": val_loss,
            "config": {
                "encoder_id": self.config.encoder_id, "llm_id": self.config.llm_id,
                "encoder_dim": self.projector.encoder_dim, "llm_dim": self.projector.llm_dim,
                "num_queries": self.projector.num_queries, "qformer_dim": self.config.qformer_dim,
                "qformer_layers": self.config.qformer_layers, "qformer_heads": self.config.qformer_heads,
                "dropout": self.config.dropout, "loss_type": "cosine_contrastive_moco",
            },
        }
        path = sd / f"projector_{tag}.pt"
        torch.save(ckpt, path); logger.info(f"  Saved: {path}")
        torch.save(torch.load(path, weights_only=False), sd / "latest.pt")
        self._cp_drive(path)
        if tag == "best": self._cp_drive(sd / "latest.pt")

    def _cp_drive(self, src):
        import shutil
        dst = Path("/content/drive/MyDrive/AnyProjector") / self.config.save_dir
        if not Path("/content/drive").exists(): return
        try: dst.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst/src.name); logger.info(f"  -> Drive: {src.name}")
        except Exception as e: logger.warning(f"  Drive copy failed: {e}")

    def load_checkpoint(self, path):
        logger.info(f"Resuming from: {path}")
        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        self.projector.load_state_dict(ckpt["projector_state_dict"], strict=False)
        if "log_temp" in ckpt: self.log_temp.data.fill_(ckpt["log_temp"])
        if self.optimizer and "optimizer_state_dict" in ckpt:
            try: self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            except: logger.warning("  Could not load optimizer state (architecture changed)")
        if self.scheduler and ckpt.get("scheduler_state_dict"):
            try: self.scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            except: pass
        self.global_step = ckpt.get("global_step", 0)
        self.start_epoch = ckpt.get("epoch", 0)
        if isinstance(self.start_epoch, str) or self.start_epoch < 0: self.start_epoch = 0
        logger.info(f"  Resumed: epoch={self.start_epoch}, step={self.global_step}")

## Train

In [ ]:
config = Phase2Config()
trainer = Phase2Trainer(config)
trainer.train(preloaded_dataset=ds)